In [1]:
# ============================================================
#  Next-Word Text Generation — Alice's Adventures in Wonderland
#  Colab GPU version (LSTM + GRU)
# ============================================================
# Runtime -> Change runtime type -> Hardware accelerator -> GPU

import re
import requests
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Confirm GPU is active
print("GPU:", tf.config.list_physical_devices('GPU'))

# ============================================================
#  1. Load data
# ============================================================
url = "https://www.gutenberg.org/cache/epub/11/pg11.txt"
raw = requests.get(url).text

# ============================================================
#  2. Preprocess
#     Goal = generate sentences, so split into lines (sentences),
#     strip Gutenberg header/footer, remove non-words.
# ============================================================
def preprocess(text):
    start = text.find('*** START')
    end   = text.find('*** END')
    text = text[start:end]
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)      # keep letters only
    text = re.sub(r'[ \t]+', ' ', text)
    corpus = [line.strip() for line in text.split('\n') if len(line.strip()) > 0]
    return corpus

corpus = preprocess(raw)
print(corpus[:5])
print(' '.join(corpus)[:200])                  # first 200 chars

# ============================================================
#  3. Tokenize + vocabulary
# ============================================================
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)
total_words = len(tokenizer.word_index) + 1
print("total_words:", total_words)

# ============================================================
#  4. Build n-gram sequences
# ============================================================
input_sequences = []
for line in corpus:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i + 1])

# Cap length so we don't pad everything to one long outlier line
max_seq_len = min(max(len(s) for s in input_sequences), 20)

# Pre-padding: the word to predict stays at the last position
input_sequences = np.array(pad_sequences(input_sequences,
                                         maxlen=max_seq_len, padding='pre'))

# Split predictors / label
X = input_sequences[:, :-1]
y = input_sequences[:, -1]
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

print("X shape:", X.shape, "| y shape:", y.shape)

# ============================================================
#  5. Build model  (single recurrent layer -> cuDNN fast path)
# ============================================================
def build_model(rnn='LSTM'):
    model = Sequential()
    model.add(Embedding(total_words, 100, input_length=max_seq_len - 1))
    if rnn == 'LSTM':
        model.add(LSTM(150))
    else:
        model.add(GRU(150))
    model.add(Dropout(0.2))
    model.add(Dense(total_words, activation='softmax'))
    return model

# ============================================================
#  6. Text generation function
# ============================================================
def generate_text(seed_text, next_words, model, max_seq_len, tokenizer):
    output = seed_text
    seed = re.sub(r'[^a-z\s]', ' ', seed_text.lower())     # same preprocessing
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed])[0]
        token_list = pad_sequences([token_list],
                                   maxlen=max_seq_len - 1, padding='pre')
        predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)[0]
        next_word = tokenizer.index_word.get(predicted, '')
        if not next_word:
            break
        seed += ' ' + next_word
        output += ' ' + next_word
    return output

early_stop = EarlyStopping(monitor='loss', patience=3,
                           restore_best_weights=True)

# ============================================================
#  7. Train + generate — LSTM
# ============================================================
print("\n===== LSTM =====")
lstm_model = build_model('LSTM')
lstm_model.compile(loss='categorical_crossentropy',
                   optimizer='adam', metrics=['accuracy'])
lstm_model.fit(X, y, epochs=50, batch_size=256,
               callbacks=[early_stop], verbose=1)

print(generate_text("alice was", 20, lstm_model, max_seq_len, tokenizer))

# ============================================================
#  8. Train + generate — GRU
# ============================================================
print("\n===== GRU =====")
gru_model = build_model('GRU')
gru_model.compile(loss='categorical_crossentropy',
                  optimizer='adam', metrics=['accuracy'])
gru_model.fit(X, y, epochs=50, batch_size=256,
              callbacks=[early_stop], verbose=1)

print(generate_text("alice was", 20, gru_model, max_seq_len, tokenizer))

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
['start of the project gutenberg ebook alice s adventures in wonderland', 'illustration', 'alice s adventures in wonderland', 'by lewis carroll', 'the millennium fulcrum edition']
start of the project gutenberg ebook alice s adventures in wonderland illustration alice s adventures in wonderland by lewis carroll the millennium fulcrum edition contents chapter i down the rabbit h
total_words: 2580
X shape: (24952, 19) | y shape: (24952, 2580)

===== LSTM =====


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/50
98/98 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.0585 - loss: 6.3838
Epoch 2/50
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.0610 - loss: 5.9060
Epoch 3/50
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.0614 - loss: 5.8057
Epoch 4/50
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.0675 - loss: 5.6974
Epoch 5/50
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.0783 - loss: 5.5709
Epoch 6/50
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.0928 - loss: 5.4449
Epoch 7/50
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.1055 - loss: 5.3283
Epoch 8/50
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.1173 - loss: 5.2260
Epoch 9/50
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.1242 - loss: 5.1280
Epoch 10/50
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.1358 - loss: 5.0324
Epoch 11/50
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.1426 - loss: 4.9439
Epoch 12/50
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.1481